In [1]:
import rasterio
from rasterio.mask import mask
import geopandas as gpd
import numpy as np
import os

In [2]:
bands = ["B02", "B03", "B04", "B08"]

os.makedirs("tehri_landslide/data/sentinel_bands/reflectance", exist_ok=True)

for band in bands:
    with rasterio.open(f"tehri_landslide/data/sentinel_bands/bands_tif/{band}_Tehri.tif") as src:
        dn = src.read(1).astype("float32")

        # mask DN NoData
        dn[dn == 65535] = np.nan

        # DN → reflectance
        refl = dn * 0.0001

        meta = src.meta.copy()
        meta.update({
            "dtype": "float32",
            "nodata": -9999
        })

        refl[np.isnan(refl)] = -9999

        with rasterio.open(f"tehri_landslide/data/sentinel_bands/reflectance/{band}_ref.tif", "w", **meta) as dst:
            dst.write(refl, 1)

print("DN → Reflectance completed for all bands")


DN → Reflectance completed for all bands


In [4]:
with rasterio.open("tehri_landslide/data/sentinel_bands/reflectance/B04_ref.tif") as src:
    arr = src.read(1)
    print("Min:", np.nanmin(arr[arr != -9999]))
    print("Max:", np.nanmax(arr[arr != -9999]))


Min: 0.0
Max: 1.7407999


In [5]:
import rasterio
import numpy as np
import os

os.makedirs("tehri_landslide/data/sentinel_bands/reflectance_clean", exist_ok=True)

bands = ["B02", "B03", "B04", "B08"]

for band in bands:
    with rasterio.open(f"tehri_landslide/data/sentinel_bands/reflectance/{band}_ref.tif") as src:
        arr = src.read(1)

        # keep NoData
        nodata = src.nodata

        # clip reflectance
        arr_clean = arr.copy()
        arr_clean[arr_clean == nodata] = np.nan
        arr_clean = np.clip(arr_clean, 0.0, 1.0)
        arr_clean[np.isnan(arr_clean)] = nodata

        meta = src.meta.copy()

        with rasterio.open(f"tehri_landslide/data/sentinel_bands/reflectance_clean/{band}_ref.tif", "w", **meta) as dst:
            dst.write(arr_clean, 1)

print("Reflectance clipped to [0, 1]")


Reflectance clipped to [0, 1]


In [6]:
with rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B04_ref.tif") as src:
    arr = src.read(1)
    print(
        "Min:", np.nanmin(arr[arr != -9999]),
        "Max:", np.nanmax(arr[arr != -9999])
    )


Min: 0.0 Max: 1.0


# NDVI

$$\text{NDVI} = \frac{\text{NIR} - \text{RED}}{\text{NIR} + \text{RED}}$$
$$\text{NDVI} = \frac{\text{B08} - \text{B04}}{\text{B08} + \text{B04}}$$

In [7]:

os.makedirs("tehri_landslide/data/sentinel_indices", exist_ok=True)

with rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B08_ref.tif") as nir, \
     rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B04_ref.tif") as red:

    nir_arr = nir.read(1)
    red_arr = red.read(1)

    # mask NoData
    nir_arr[nir_arr == -9999] = np.nan
    red_arr[red_arr == -9999] = np.nan

    # NDVI calculation
    ndvi = (nir_arr - red_arr) / (nir_arr + red_arr)

    # clean numerical issues
    ndvi[np.isinf(ndvi)] = np.nan

    meta = nir.meta.copy()
    meta.update({
        "dtype": "float32",
        "nodata": -9999
    })

    ndvi[np.isnan(ndvi)] = -9999

    with rasterio.open("tehri_landslide/data/sentinel_indices/NDVI_Tehri.tif", "w", **meta) as dst:
        dst.write(ndvi, 1)

print("NDVI_Tehri.tif created")


C:\Users\sangh\AppData\Local\Temp\ipykernel_6960\2964964000.py:14: RuntimeWarning: invalid value encountered in divide
  ndvi = (nir_arr - red_arr) / (nir_arr + red_arr)


NDVI_Tehri.tif created


In [8]:
with rasterio.open("tehri_landslide/data/sentinel_indices/NDVI_Tehri.tif") as src:
    ndvi = src.read(1)
    print("NDVI min:", np.nanmin(ndvi[ndvi != -9999]))
    print("NDVI max:", np.nanmax(ndvi[ndvi != -9999]))


NDVI min: -0.99799997
NDVI max: 0.9993666


# NDWI

$$\text{NDWI} = \frac{\text{GREEN} - \text{NIR}}{\text{GREEN} + \text{NIR}}$$

In [9]:
os.makedirs("tehri_landslide/data/sentinel_indices", exist_ok=True)

with rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B03_ref.tif") as green, \
     rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B08_ref.tif") as nir:

    green_arr = green.read(1)
    nir_arr = nir.read(1)

    # mask NoData
    green_arr[green_arr == -9999] = np.nan
    nir_arr[nir_arr == -9999] = np.nan

    # NDWI calculation
    ndwi = (green_arr - nir_arr) / (green_arr + nir_arr)

    # clean numerical issues
    ndwi[np.isinf(ndwi)] = np.nan

    meta = green.meta.copy()
    meta.update({
        "dtype": "float32",
        "nodata": -9999
    })

    ndwi[np.isnan(ndwi)] = -9999

    with rasterio.open("tehri_landslide/data/sentinel_indices/NDWI_Tehri.tif", "w", **meta) as dst:
        dst.write(ndwi, 1)

print("NDWI_Tehri.tif created")

with rasterio.open("tehri_landslide/data/sentinel_indices/NDWI_Tehri.tif") as src:
    ndwi = src.read(1)
    print("NDWI min:", np.nanmin(ndwi[ndwi != -9999]))
    print("NDWI max:", np.nanmax(ndwi[ndwi != -9999]))

C:\Users\sangh\AppData\Local\Temp\ipykernel_6960\669209921.py:14: RuntimeWarning: invalid value encountered in divide
  ndwi = (green_arr - nir_arr) / (green_arr + nir_arr)


NDWI_Tehri.tif created
NDWI min: -0.99881446
NDWI max: 0.99808425


# BSI (Bare Soil Index) Lite

$$\text{BSI}_{\text{lite}} = \frac{(\text{RED} + \text{GREEN}) + \text{NIR}}{(\text{RED} + \text{GREEN}) - \text{NIR}}$$

In [10]:

with rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B03_ref.tif") as green, \
     rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B04_ref.tif") as red, \
     rasterio.open("tehri_landslide/data/sentinel_bands/reflectance_clean/B08_ref.tif") as nir:
    g = green.read(1)
    r = red.read(1)
    n = nir.read(1)

    # mask NoData
    g[g == -9999] = np.nan
    r[r == -9999] = np.nan
    n[n == -9999] = np.nan

    # BSI-lite
    bsi = ((r + g) - n) / ((r + g) + n)

    bsi[np.isinf(bsi)] = np.nan

    meta = green.meta.copy()
    meta.update({
        "dtype": "float32",
        "nodata": -9999
    })

    bsi[np.isnan(bsi)] = -9999

    with rasterio.open("tehri_landslide/data/sentinel_indices/BSI_lite_Tehri.tif", "w", **meta) as dst:
        dst.write(bsi, 1)

print("BSI-lite created")

C:\Users\sangh\AppData\Local\Temp\ipykernel_6960\72707781.py:14: RuntimeWarning: invalid value encountered in divide
  bsi = ((r + g) - n) / ((r + g) + n)


BSI-lite created


In [11]:
with rasterio.open("tehri_landslide/data/sentinel_indices/BSI_lite_Tehri.tif") as src:
    arr = src.read(1)
    print("BSI-lite min:", np.nanmin(arr[arr != -9999]))
    print("BSI-lite max:", np.nanmax(arr[arr != -9999]))


BSI-lite min: -0.5048366
BSI-lite max: 0.99902105
